In [24]:
import pandas as pd
import numpy as np
import yaml

from preproc_paths import ( 
    DEFAULT_SHOTS_STATS_TRAIN_FILE, 
    DEFAULT_SHOTS_STATS_VAL_FILE, 
    DEFAULT_SHOTS_STATS_TEST_FILE,
    DEFAULT_SIGNALS_MEAN_STD_TRAIN_FILE
)

## Get global mean and std for stdscaling

In [25]:
df_train = pd.read_csv(DEFAULT_SHOTS_STATS_TRAIN_FILE).sort_values(by=["shot_idx", "variable"])

In [26]:
df_train[df_train["shot_id"]==19454].head()

,shot_idx,shot_id,variable,n_dim_shot,mean,variance
230170,5901,19454,equilibrium-beta_normal,106.0,-9.177884,80.380965
230169,5901,19454,equilibrium-beta_pol,106.0,-1.818459,2.065338
230168,5901,19454,equilibrium-beta_tor,106.0,-16.315314,305.082886
230172,5901,19454,equilibrium-bphi_rmag,106.0,-1.056248,0.212465
230171,5901,19454,equilibrium-bvac_rmag,106.0,-0.538247,0.004125


In [27]:
# GLOBAL MEAN BASED ON TRAIN
global_mean = (
    df_train
    .groupby("variable")[["n_dim_shot", "mean", ]]
    .apply(lambda g: (g["mean"] * g["n_dim_shot"]).sum() / g["n_dim_shot"].sum())
    .rename("global_mean")
)
df_train_with_group_mean = df_train.join(global_mean, on="variable")

# GLOBAL VARIANCE BASED ON TRAIN
global_variance = (
    df_train_with_group_mean
    .groupby("variable")[["n_dim_shot", "mean", "variance", "global_mean"]]
    .apply(lambda g: ( g["n_dim_shot"] * g["variance"] + g["n_dim_shot"] * (g["mean"]-g["global_mean"])**2 ).sum() / g["n_dim_shot"].sum())
    .rename("global_variance")
)
df_train_with_group_mean_and_variance = df_train_with_group_mean.join(global_variance, on="variable")
df_train_with_group_mean_and_variance.head()

,shot_idx,shot_id,variable,n_dim_shot,mean,variance,global_mean,global_variance
31,0,21719,equilibrium-beta_normal,109.0,0.509525,0.157241,0.984934,3.284780
30,0,21719,equilibrium-beta_pol,109.0,0.118568,0.018020,0.206842,0.152316
29,0,21719,equilibrium-beta_tor,109.0,1.070841,0.414899,2.949390,225.456765
33,0,21719,equilibrium-bphi_rmag,109.0,-0.554288,0.001070,-0.513734,0.040348
32,0,21719,equilibrium-bvac_rmag,109.0,-0.474535,0.001585,-0.444512,0.028251


#### Remove z-6 outliers from computation

In [28]:
# Z-SCORE COMPUTATION TRAIN
# df_train_with_group_mean_and_variance["z_score"] = (df_train_with_group_mean_and_variance["mean"] - df_train_with_group_mean_and_variance["global_mean"]) / ( np.sqrt(df_train_with_group_mean_and_variance["global_variance"]) / np.sqrt(df_train_with_group_mean_and_variance["n_dim_shot"]) )
df_train_with_group_mean_and_variance["z_score"] = (df_train_with_group_mean_and_variance["mean"] - df_train_with_group_mean_and_variance["global_mean"]) / ( np.sqrt(df_train_with_group_mean_and_variance["global_variance"]) )

df_train_with_group_mean_and_variance

# COUNTING OUTLIERS
print( f'Count of z-6 outliers {sum ( abs(df_train_with_group_mean_and_variance["z_score"]) > 6 )} out of {len(df_train_with_group_mean_and_variance)}')
print( f'Count of z-12 outliers {sum ( abs(df_train_with_group_mean_and_variance["z_score"]) > 12 )} out of {len(df_train_with_group_mean_and_variance)}')

df_train_with_group_mean_and_variance["outlier_z_6"] = abs(df_train_with_group_mean_and_variance["z_score"]) > 6 
df_train_with_group_mean_and_variance["outlier_z_12"] = abs(df_train_with_group_mean_and_variance["z_score"]) > 12

df_train_extended = df_train_with_group_mean_and_variance.copy()
df_train_with_group_mean_and_variance.sort_values('z_score').dropna()

Count of z-6 outliers 144 out of 361101
Count of z-12 outliers 41 out of 361101


,shot_idx,shot_id,variable,n_dim_shot,mean,variance,global_mean,global_variance,z_score,outlier_z_6,outlier_z_12
29242,749,19382,equilibrium-beta_normal,90.0,-6.262440e+01,8.627237e+02,9.849342e-01,3.284780e+00,-35.096814,True,True
29241,749,19382,equilibrium-beta_pol,90.0,-1.011090e+01,2.021253e+01,2.068420e-01,1.523160e-01,-26.436981,True,True
284887,7304,19374,equilibrium-beta_normal,90.0,-4.281552e+01,1.528744e+02,9.849342e-01,3.284780e+00,-24.167153,True,True
284886,7304,19374,equilibrium-beta_pol,90.0,-6.496856e+00,2.540111e+00,2.068420e-01,1.523160e-01,-17.176778,True,True
106072,2719,19375,equilibrium-beta_normal,77.0,-2.886174e+01,1.026092e+01,9.849342e-01,3.284780e+00,-16.468073,True,True
...,...,...,...,...,...,...,...,...,...,...,...
5801,148,12214,equilibrium-beta_tor,120.0,2.721943e+02,2.744358e+04,2.949390e+00,2.254568e+02,17.931472,True,True
164141,4208,12234,equilibrium-beta_tor,125.0,2.829469e+02,3.657616e+04,2.949390e+00,2.254568e+02,18.647580,True,True
233834,5995,12223,equilibrium-beta_tor,122.0,2.848120e+02,3.661117e+04,2.949390e+00,2.254568e+02,18.771799,True,True
158174,4055,12232,equilibrium-beta_tor,124.0,3.119264e+02,4.274548e+04,2.949390e+00,2.254568e+02,20.577591,True,True


In [29]:
# import matplotlib.pyplot as plt

# plt.hist(df_train_with_group_mean_and_variance["z_score"], bins=30)
# plt.xlabel("z_score")
# plt.ylabel("Frequency")
# plt.title("Histogram of z-scores")
# plt.show()

In [30]:
df_train_extended.head()

,shot_idx,shot_id,variable,n_dim_shot,mean,variance,global_mean,global_variance,z_score,outlier_z_6,outlier_z_12
31,0,21719,equilibrium-beta_normal,109.0,0.509525,0.157241,0.984934,3.284780,-0.262310,False,False
30,0,21719,equilibrium-beta_pol,109.0,0.118568,0.018020,0.206842,0.152316,-0.226183,False,False
29,0,21719,equilibrium-beta_tor,109.0,1.070841,0.414899,2.949390,225.456765,-0.125110,False,False
33,0,21719,equilibrium-bphi_rmag,109.0,-0.554288,0.001070,-0.513734,0.040348,-0.201892,False,False
32,0,21719,equilibrium-bvac_rmag,109.0,-0.474535,0.001585,-0.444512,0.028251,-0.178628,False,False


In [31]:
# OG global mean and variance
global_mean = (
    df_train_extended
    .groupby("variable")[["n_dim_shot", "mean"]]
    .apply(lambda g: (g["mean"] * g["n_dim_shot"]).sum() / g["n_dim_shot"].sum())
    .rename("global_mean")
)
global_variance = (
    df_train_extended
    .groupby("variable")[["n_dim_shot", "mean", "variance", "global_mean"]]
    .apply(lambda g: ( g["n_dim_shot"] * g["variance"] + g["n_dim_shot"] * (g["mean"]-g["global_mean"])**2 ).sum() / g["n_dim_shot"].sum())
    .rename("global_variance")
)

# MEAN AND VARIANCE COMPUTATION WITHOUT OUTLIERS Z-6
# global mean and variance without the 6-z outliers
global_mean_no_z_6 = (
    df_train_extended[~df_train_extended['outlier_z_6']]
    .groupby("variable")[["n_dim_shot", "mean"]]
    .apply(lambda g: (g["mean"] * g["n_dim_shot"]).sum() / g["n_dim_shot"].sum())
    .rename("global_mean_no_z_6")
)

df_train_extended = df_train_extended.join(global_mean_no_z_6, on="variable")
global_variance_no_z_6 = (
    df_train_extended[~df_train_extended['outlier_z_6']]
    .groupby("variable")[["n_dim_shot", "mean", "variance", "global_mean_no_z_6"]]
    .apply(lambda g: ( g["n_dim_shot"] * g["variance"] + g["n_dim_shot"] * (g["mean"]-g["global_mean_no_z_6"])**2 ).sum() / g["n_dim_shot"].sum())
    .rename("global_variance_no_z_6")
)


In [34]:
# Merge all the stats into a single DataFrame
df_stats = pd.DataFrame({
    "variable": global_mean.index,
    # "mean_all": global_mean.values,
    # "std_all": np.sqrt(global_variance.values),
    "mean_no_outliers_z6": global_mean_no_z_6.values,
    "std_no_outliers_z6": np.sqrt(global_variance_no_z_6.values),
})

# Build the dictionary for YAML
final_dict = {}
for _, row in df_stats.iterrows():
    var = row["variable"]
    final_dict[var] = {
        "mean": {
            # "all": row["mean_all"],
            "no_outliers_z6": row["mean_no_outliers_z6"],
            # "no_outliers_z12": row["mean_no_outliers_z12"]
        },
        "std": {
            # "all": row["std_all"],
            "no_outliers_z6": row["std_no_outliers_z6"],
            # "no_outliers_z12": row["std_no_outliers_z12"]
        }
    }

# Write to YAML
with open(DEFAULT_SIGNALS_MEAN_STD_TRAIN_FILE, "w") as f:
    yaml.dump(final_dict, f, sort_keys=False)


## Remove outliers from train, val, test

In [22]:
df_train = pd.read_csv(DEFAULT_SHOTS_STATS_TRAIN_FILE).sort_values(by=["shot_idx", "variable"])
df_val = pd.read_csv(DEFAULT_SHOTS_STATS_VAL_FILE).sort_values(by=["shot_idx", "variable"])
df_test = pd.read_csv(DEFAULT_SHOTS_STATS_TEST_FILE).sort_values(by=["shot_idx", "variable"])

df_all = pd.concat([df_train, df_val, df_test])

In [23]:
df_all_with_stats = df_all.join(global_mean_no_z_6, on="variable").join(global_variance_no_z_6, on="variable")
df_all_with_stats

df_all_with_stats["z_score"] = (df_all_with_stats["mean"] - df_all_with_stats["global_mean_no_z_6"]) / ( np.sqrt(df_all_with_stats["global_variance_no_z_6"]) )

df_all_with_stats["outlier_z_12"] = abs(df_all_with_stats["z_score"]) > 12

print( f'Count of z-12 outliers {sum ( df_all_with_stats["outlier_z_12"] ) } out of {len(df_all_with_stats)}')
print( f'Spanning {len( df_all_with_stats[ df_all_with_stats["outlier_z_12"]==True ]["variable"].unique() )} variables {df_all_with_stats[ df_all_with_stats["outlier_z_12"]==True ]["variable"].unique() } (out of {len( df_all_with_stats["variable"].unique()) })' )
print( f'Spanning {len( df_all_with_stats[ df_all_with_stats["outlier_z_12"]==True ]["shot_id"].unique() )} shots {df_all_with_stats[ df_all_with_stats["outlier_z_12"]==True ]["shot_id"].unique()} (out of {len( df_all_with_stats["shot_id"].unique()) })' )


Count of z-12 outliers 141 out of 451347
Spanning 8 variables ['equilibrium-x_point_r' 'equilibrium-beta_tor' 'pf_active-coil_voltage'
 'soft_x_rays-horizontal_cam_upper' 'equilibrium-beta_normal'
 'equilibrium-beta_pol' 'equilibrium-bphi_rmag' 'thomson_scattering-n_e'] (out of 39)
Spanning 117 shots [12198 12142 12214 25456 12143 12228 13298 19382 19391 25454 12236 12184
 12224 12202 12178 12165 12179 12210 12145 20111 12221 12162 12196 25455
 12170 12164 15926 19415 19375 12180 12161 19461 12200 12150 20166 12139
 12185 12158 25453 12232 12216 12233 12234 12219 12209 12199 19953 12191
 12163 12230 12218 12148 19417 20038 12053 12177 12175 12213 12144 20271
 19413 19409 19454 12223 12195 19453 12231 12171 12168 12181 19393 12220
 19432 12146 19374 12215 12166 12190 12147 12237 19419 12208 19414 12169
 12151 12204 19460 20152 20124 12193 12201 19450 12194 12173 12182 12172
 12140 19352 19401 19420 12183 12189 19388 19386 12229 12174 12154 12130
 12167 25457 12141 12203 19410 19455 1938

In [24]:
# import matplotlib.pyplot as plt

# plt.hist(
#     df_all_with_stats[df_all_with_stats["outlier_z_12"]]["shot_id"],
#     bins=100,
#     color="salmon",
#     edgecolor="black"
# )
# plt.xlabel("shot_id")
# plt.ylabel("Number of outliers")
# plt.show()


In [25]:
outlier_dict = (
    df_all_with_stats[df_all_with_stats["outlier_z_12"]].groupby("shot_id")["variable"]
    .apply(list)
    .to_dict()
)
print(outlier_dict)

with open("dict_outlier_metadata.yaml", "w") as f_:
    yaml.dump(outlier_dict, f_, sort_keys=False)

{12053: ['equilibrium-x_point_r'], 12130: ['equilibrium-x_point_r'], 12139: ['equilibrium-x_point_r'], 12140: ['equilibrium-x_point_r'], 12141: ['equilibrium-x_point_r'], 12142: ['equilibrium-x_point_r'], 12143: ['equilibrium-x_point_r'], 12144: ['equilibrium-x_point_r'], 12145: ['equilibrium-x_point_r'], 12146: ['equilibrium-x_point_r'], 12147: ['equilibrium-x_point_r'], 12148: ['equilibrium-x_point_r'], 12150: ['equilibrium-x_point_r'], 12151: ['equilibrium-x_point_r'], 12152: ['equilibrium-x_point_r'], 12154: ['equilibrium-x_point_r'], 12158: ['equilibrium-x_point_r'], 12161: ['equilibrium-x_point_r'], 12162: ['equilibrium-x_point_r'], 12163: ['equilibrium-x_point_r'], 12164: ['equilibrium-x_point_r'], 12165: ['equilibrium-x_point_r'], 12166: ['equilibrium-x_point_r'], 12167: ['equilibrium-x_point_r'], 12168: ['equilibrium-x_point_r'], 12169: ['equilibrium-x_point_r'], 12170: ['equilibrium-x_point_r'], 12171: ['equilibrium-x_point_r'], 12172: ['equilibrium-x_point_r'], 12173: ['equi

In [26]:
outlier_dict.keys()

dict_keys([12053, 12130, 12139, 12140, 12141, 12142, 12143, 12144, 12145, 12146, 12147, 12148, 12150, 12151, 12152, 12154, 12158, 12161, 12162, 12163, 12164, 12165, 12166, 12167, 12168, 12169, 12170, 12171, 12172, 12173, 12174, 12175, 12177, 12178, 12179, 12180, 12181, 12182, 12183, 12184, 12185, 12189, 12190, 12191, 12193, 12194, 12195, 12196, 12198, 12199, 12200, 12201, 12202, 12203, 12204, 12208, 12209, 12210, 12213, 12214, 12215, 12216, 12218, 12219, 12220, 12221, 12223, 12224, 12228, 12229, 12230, 12231, 12232, 12233, 12234, 12236, 12237, 13298, 15926, 19352, 19374, 19375, 19382, 19386, 19387, 19388, 19391, 19392, 19393, 19401, 19409, 19410, 19413, 19414, 19415, 19417, 19419, 19420, 19432, 19450, 19453, 19454, 19455, 19460, 19461, 19953, 20038, 20111, 20124, 20152, 20166, 20271, 25453, 25454, 25455, 25456, 25457])